In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
import pickle
import tempfile
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- io_parquet ---

print("✅ Fixtures loaded")

In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_io_parquet():
    def write_df_to_pickle(
        df: pd.DataFrame,
        path: Path,
    ) -> None:
        """Write a Pandas DataFrame to a pickle file."""
        df.to_pickle(path)

    def load_df_from_pickle(path: Path) -> pd.DataFrame:
        """Load a Pandas DataFrame from a pickle file."""
        return pd.read_pickle(path)
    return write_df_to_pickle, load_df_from_pickle


In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_io_parquet():
    import pickle
    from pathlib import Path



    def write_df_to_pickle(
        df: pl.DataFrame,
        path: Path,
    ) -> None:
        """Write a Pandas DataFrame to a pickle file."""
        with open(path, "wb") as f:
            pickle.dump(df, f)


    def load_df_from_pickle(path: Path) -> pl.DataFrame:
        """Load a Pandas DataFrame from a pickle file."""
        with open(path, "rb") as f:
            obj = pickle.load(f)
        if isinstance(obj, pl.DataFrame):
            return obj
        return pl.from_pandas(obj)
    return write_df_to_pickle, load_df_from_pickle


In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: io_parquet ===
import tempfile
try:
    _r = gen_io_parquet(); assert isinstance(_r, tuple) and len(_r) == 2
    print("✅ L1 smoke gen_io_parquet: OK, type= tuple")
except Exception as _e: print(f"❌ L1 smoke gen_io_parquet: {type(_e).__name__}: {_e}")
try:
    _rb = before_io_parquet(); assert isinstance(_rb, tuple) and len(_rb) == 2
    print("✅ L1 smoke before_io_parquet: OK")
except Exception as _e: print(f"❌ L1 smoke before_io_parquet: {type(_e).__name__}: {_e}")
try:
    _bw,_bl=before_io_parquet(); _gw,_gl=gen_io_parquet()
    with tempfile.TemporaryDirectory() as _td:
        _bp=Path(_td)/"before.pkl"; _gp=Path(_td)/"gen.ipc"
        _bdf=pd.DataFrame({"doc_id":[1,2],"score":[0.5,0.75]}); _gdf=pl.from_pandas(_bdf)
        _bw(_bdf,_bp); _gw(_gdf,_gp)
        compare(_bl(_bp),_gl(_gp),"io_parquet writer-loader roundtrip",check_row_order=True)
except Exception as _e: print(f"❌ L2 equivalence io_parquet: setup error — {type(_e).__name__}: {_e}")
try:
    _bw,_bl=before_io_parquet(); _gw,_gl=gen_io_parquet()
    with tempfile.TemporaryDirectory() as _td:
        _bp=Path(_td)/"before.pkl"; _gp=Path(_td)/"gen.ipc"
        _bdf=pd.DataFrame({"doc_id":[1,1,2],"score":[0.5,np.nan,0.75]}); _gdf=pl.DataFrame({"doc_id":[1,1,2],"score":[0.5,None,0.75]})
        _bw(_bdf,_bp); _gw(_gdf,_gp)
        compare(_bl(_bp),_gl(_gp),"L3 edge io_parquet null duplicate roundtrip",check_row_order=True)
except Exception as _e: print(f"❌ L3 edge io_parquet: {type(_e).__name__}: {_e}")
